In [13]:
import polars as pl
pl.Config.set_tbl_width_chars(200)

polars.config.Config

In [14]:
DIR_PATH = "../data/raw_data/4721.csv"
cnae_raw = pl.read_csv(DIR_PATH, 
                 separator=";").filter(
                     pl.col("Municipios").is_not_null() &
                     pl.col("Grupos CNAE").str.contains("F Construcción")
                 ).with_columns(
                     pl.col("Provincias").str.extract(r"\d+{2}", 0).alias("CODE_PROV"),
                     pl.col("Municipios").str.extract(r"\d+{5}", 0).alias("MUN_CODE"),
                     pl.col("Provincias").str.replace(r"\d+{2}\s", ""),
                     pl.col("Municipios").str.replace(r"\d+{5}\s", ""),
                     pl.col("Total")
                        .str.replace_all(r"\.", "")
                        .str.strip_chars()
                        .replace({
                            "": "0"
                        }).cast(pl.Float16)
                 ).drop("Totales Territoriales", "Grupos CNAE")


In [15]:
cnae_clean = cnae_raw.filter(
    (pl.col("Municipios") == 'Barcelona')
)
cnae_clean

Provincias,Municipios,Periodo,Total,CODE_PROV,MUN_CODE
str,str,i64,f16,str,str
"""Barcelona""","""Barcelona""",2025,12128.0,"""08""","""08019"""
"""Barcelona""","""Barcelona""",2024,12048.0,"""08""","""08019"""
"""Barcelona""","""Barcelona""",2023,11600.0,"""08""","""08019"""
"""Barcelona""","""Barcelona""",2022,14640.0,"""08""","""08019"""
"""Barcelona""","""Barcelona""",2021,14520.0,"""08""","""08019"""
…,…,…,…,…,…
"""Barcelona""","""Barcelona""",2016,14272.0,"""08""","""08019"""
"""Barcelona""","""Barcelona""",2015,14192.0,"""08""","""08019"""
"""Barcelona""","""Barcelona""",2014,14504.0,"""08""","""08019"""


In [16]:
cnae = cnae_raw.pivot(
    index=["Municipios", "MUN_CODE"],
    columns="Periodo",
    values="Total"
)
cnae

/var/folders/8z/b59c9z0d11g7v_c4yjht5glr0000gn/T/ipykernel_33389/3259643809.py:1: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  cnae = cnae_raw.pivot(


Municipios,MUN_CODE,2025,2024,2023,2022,2021,2020,2019,2018,2017,2016,2015,2014,2013,2012
str,str,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16
"""Alegría-Dulantzi""","""01001""",29.0,30.0,31.0,31.0,27.0,28.0,26.0,25.0,33.0,30.0,31.0,29.0,24.0,22.0
"""Amurrio""","""01002""",88.0,89.0,98.0,111.0,109.0,101.0,95.0,92.0,84.0,93.0,93.0,94.0,98.0,102.0
"""Aramaio""","""01003""",4.0,4.0,4.0,5.0,4.0,5.0,7.0,6.0,6.0,8.0,8.0,7.0,7.0,10.0
"""Artziniega""","""01004""",13.0,14.0,13.0,13.0,14.0,12.0,15.0,16.0,15.0,18.0,18.0,20.0,20.0,22.0
"""Armiñón""","""01006""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Biel""","""50901""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""Marracos""","""50902""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""Villamayor de Gállego""","""50903""",19.0,18.0,18.0,21.0,19.0,19.0,18.0,19.0,14.0,16.0,16.0,19.0,25.0,23.0


In [17]:
cnae.write_csv("../data/clean_data/cenae_construccion_mun_es.csv")